# ASSIGNMENT 1 - Question 7 - Small Data Chest X-Ray Classification
### Alumno: Martín Andrés Gutiérrez López
Topics Data Augmentation, Transfer Learning, backbone freezing, full-network fine-tuning

In [4]:
# ==============================================================================
# Question 7: Small-Data Chest X-Ray Classification (Case: MediScan)
# Transfer Learning, Freezing, Two-Stage Training Setup, and Parameter Audit
# ==============================================================================

import torch
import torch.nn as nn
from torchvision import models, transforms

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (a): Data Augmentation Pipeline for Chest X-Rays
# ------------------------------------------------------------------------------
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    # Small random rotation to account for patient orientation (no vertical/horizontal flips)
    transforms.RandomRotation(degrees=10),
    # Conservative random crop to preserve lung field anatomy
    transforms.RandomResizedCrop(size=224, scale=(0.85, 1.0)),
    # Mild contrast/brightness adjustment to mimic X-ray sensor variations
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    # Standard ImageNet normalization stats
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (a): AUGMENTATION PIPELINE DEFINED")
print("=" * 75,f"{RESET}")
print(
    "Inappropriate Augmentations: Vertical Flip (inverts anatomy/fluid levels)"
)
print("                            Horizontal Flip (alters cardiac/situs axis)")
print()

# ------------------------------------------------------------------------------
# Part (b): Model Instantiation, Backbone Freezing, and Head Replacement
# ------------------------------------------------------------------------------
# Load ImageNet-pretrained ResNet-50
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Step 1: Freeze all backbone parameters
for param in model.parameters():
  param.requires_grad = False

# Step 2: Replace classification head for 3 classes (normal, pneumonia, TB)
num_features = model.fc.in_features  # 2048
model.fc = nn.Linear(in_features=num_features, out_features=3, bias=True)

# Parameter audit
total_params = sum(p.numel() for p in model.parameters())
trainable_params_stage1 = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
frozen_params = total_params - trainable_params_stage1

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (b): CLASSIFIER HEAD REPLACEMENT & PARAMETER AUDIT")
print("=" * 75,f"{RESET}")
print(f"Total Model Parameters         : {total_params:,}")
print(f"Frozen Backbone Parameters     : {frozen_params:,}")
print(f"Trainable Parameters (Head)   : {trainable_params_stage1:,} (6144 weights + 3 biases)")
print()


# ------------------------------------------------------------------------------
# Part (c): Two-Stage Training Setup and BatchNorm Configuration
# ------------------------------------------------------------------------------
def freeze_batchnorm_layers(module):
  """Ensures all BatchNorm layers stay in eval mode during fine-tuning."""
  if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
    module.eval()


# STAGE 1 OPTIMIZER: Train only the new head
optimizer_stage1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,  # Base learning rate for new classification head
    weight_decay=1e-4,
)

# STAGE 2 SETUP: Unfreeze deep residual blocks for fine-tuning
# Unfreeze layer4 and classification head
for param in model.layer4.parameters():
  param.requires_grad = True

trainable_params_stage2 = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

# Differential learning rates for fine-tuning
optimizer_stage2 = torch.optim.Adam(
    [
        {"params": model.layer4.parameters(), "lr": 1e-5},  # Low LR for feature extractor
        {"params": model.fc.parameters(), "lr": 1e-4},  # Moderate LR for classifier head
    ],
    weight_decay=1e-4,
)

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (c): TWO-STAGE TRAINING CONFIGURATION")
print("=" * 75,f"{RESET}")
print(f"Stage 1 Trainable Parameters   : {trainable_params_stage1:,} (lr = 1e-3)")
print(f"Stage 2 Trainable Parameters   : {trainable_params_stage2:,} (layer4 lr = 1e-5, fc lr = 1e-4)")
print("BatchNorm Policy in Stage 1/2  : Frozen in eval mode to prevent population statistic drift")
print("-" * 75)

# Verify execution with a dummy input batch (Batch=4, Channels=3, Height=224, Width=224)
dummy_xray = torch.randn(4, 3, 224, 224)
logits = model(dummy_xray)
print(f"Output Logits Shape            : {list(logits.shape)} (Batch=4, Classes=3)")
print("")



PART (a): AUGMENTATION PIPELINE DEFINED
Inappropriate Augmentations: Vertical Flip (inverts anatomy/fluid levels)
                            Horizontal Flip (alters cardiac/situs axis)



PART (b): CLASSIFIER HEAD REPLACEMENT & PARAMETER AUDIT
Total Model Parameters         : 23,514,179
Frozen Backbone Parameters     : 23,508,032
Trainable Parameters (Head)   : 6,147 (6144 weights + 3 biases)



PART (c): TWO-STAGE TRAINING CONFIGURATION
Stage 1 Trainable Parameters   : 6,147 (lr = 1e-3)
Stage 2 Trainable Parameters   : 14,970,883 (layer4 lr = 1e-5, fc lr = 1e-4)
BatchNorm Policy in Stage 1/2  : Frozen in eval mode to prevent population statistic drift
---------------------------------------------------------------------------
Output Logits Shape            : [4, 3] (Batch=4, Classes=3)

